# 06a · Deep analysis of a trained PPO agent

Notebook **06** drives the RL stack end-to-end on a *tiny* budget — it proves the
mechanism (train → curves → CRN eval) in a couple of minutes, but the agent barely learns
and is expected to *lose* to the baseline. This notebook is the read-only companion: it
**loads a real, long-trained run from disk** and analyzes it in depth, re-running no
training. It pairs with 06 the same way **05a** pairs with **05**.

What it does, all offline against the artifacts a run leaves under `runs/<experiment>/`:

1. **Training trajectory** — every scalar in the TensorBoard event file, including the full
   `eval/*` history over the whole run (not the two noisy points a tiny demo produces).
2. **Model selection** — the checkpoint with the best *held-out* `eval/paired_uplift` is
   often not the last one. We pick it explicitly and analyze *that* policy.
3. **Deep CRN comparison vs the `OrderUpToPolicy` default** on a **larger** held-out seed
   set than training used — aggregate KPIs, bootstrap CI on the uplift, win rate, and the
   per-seed picture.
4. **Regime analysis** — where the policy wins or loses across two orders of magnitude of
   node *capacity* and opening *balance* (does the scale-invariance hold?).
5. **Behavioral signature** — how the trained policy differs from the textbook rule on
   stockouts, turnover, and pricing.

### Produce a run to analyze

This notebook needs a real run on disk. Train one with the CLI (synthetic catalog — no
LLM, no GPU, no API key). A serious run is ~1M env steps; budget more if you have the time
— the eval cadence below writes a checkpoint every 50k steps, so a 2M-step run leaves ~40
checkpoints to pick from:

```bash
uv run python -m src.rl.train \
  --total-env-steps 2000000 \
  --n-envs 8 \
  --episode-length 180 \
  --k-active 5 \
  --k-catalog 100 \
  --eval-cadence-env-steps 50000 \
  --n-eval-seeds 32 \
  --experiment-name ppo_long
```

Watch it live with `uv run tensorboard --logdir runs/` (the headline curve is
`eval/paired_uplift`); it's safe to stop early once that plateaus — the checkpoints already
written stay valid. Then point `RUN_DIR` below at `runs/ppo_long/`.

In [ ]:
%matplotlib inline
import os
from pathlib import Path

# Hop up to the repo root so `import src...` and relative paths resolve.
while not (Path.cwd() / "pyproject.toml").exists():
    os.chdir("..")

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 90
print("cwd:", Path.cwd())

## 1 · Point at a run

Set `RUN_DIR` to the run you trained, and restate the knobs you trained with — `train.py`
writes scalars and checkpoints but **not** the `RLConfig`, so the eval below has to rebuild
a matching one. The defaults here match the recommended command above. `K_active` is the
one knob we can recover from the checkpoint itself (it fixes the network's input width), so
we **auto-detect and assert** it as a guard against a mismatched config.

In [ ]:
RUN_DIR        = Path("runs/ppo_long")   # <- the run you trained
K_ACTIVE       = 5                        # must match --k-active
K_CATALOG      = 100                      # must match --k-catalog
EPISODE_LENGTH = 180                      # must match --episode-length
N_EVAL_SEEDS   = 64                       # held-out seeds for the deep eval (>= training's)

assert RUN_DIR.is_dir(), (
    f"no such run dir: {RUN_DIR}. Train one first (see the command at the top), "
    "then point RUN_DIR at runs/<experiment-name>/."
)

event_files = sorted(RUN_DIR.glob("events.out.tfevents.*"))
ckpts       = sorted((RUN_DIR / "checkpoints").glob("actor_step*.pt"))
assert event_files, f"no TensorBoard event file under {RUN_DIR}"
assert ckpts, f"no checkpoints under {RUN_DIR}/checkpoints"

print(f"run: {RUN_DIR}")
print(f"event file: {event_files[0].name}")
print(f"{len(ckpts)} checkpoints:")
shown = ckpts if len(ckpts) <= 6 else ckpts[:3] + ckpts[-3:]
for i, p in enumerate(shown):
    if len(ckpts) > 6 and i == 3:
        print("  ...")
    print(f"  {p.name}  ({p.stat().st_size / 1024:.0f} KB)")

The checkpoint filenames are zero-padded so they sort chronologically: the eval-cadence
checkpoints come first (named by eval index `0, 1, 2, …`), and the **final** checkpoint —
named by the total env-step count — sorts last. We verify `K_active` against the network's
input layer (`obs_dim = K_active * 18 + 4`).

In [ ]:
from src.rl.encoders import observation_dim, action_dim

# Recover obs_dim from the first Linear layer of any checkpoint: net.0.weight is
# (hidden_size, obs_dim). Then K_active = (obs_dim - 4) // 18  (N_PER_SKU=18, N_GLOBAL=4).
sd = torch.load(ckpts[-1], map_location="cpu")
obs_dim_ckpt = sd["net.0.weight"].shape[1]
k_active_ckpt = (obs_dim_ckpt - 4) // 18

assert k_active_ckpt == K_ACTIVE, (
    f"checkpoint was trained with K_active={k_active_ckpt} but K_ACTIVE={K_ACTIVE}. "
    "Fix K_ACTIVE to match the run."
)
print(f"checkpoint obs_dim = {obs_dim_ckpt}  ->  K_active = {k_active_ckpt}  (matches) ✓")

## 2 · The training trajectory

The event file holds the same scalars TensorBoard shows. On a real run the curves are long
enough to actually read: `losses/entropy` drifts down as the policy sharpens,
`train/episodic_return` trends up, and the `eval/*` series tells you whether that translated
into held-out performance. We **print the available tags first** and plot whichever exist.

In [ ]:
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(str(event_files[0]))
ea.Reload()
scalar_tags = ea.Tags()["scalars"]
print(f"{len(scalar_tags)} scalar tags logged.")

def series(tag):
    "Return (steps, values) arrays for a scalar tag, or ([], []) if absent."
    if tag not in scalar_tags:
        return np.array([]), np.array([])
    evs = ea.Scalars(tag)
    return np.array([e.step for e in evs]), np.array([e.value for e in evs])

In [ ]:
panel_tags = [
    "train/episodic_return", "losses/policy_loss", "losses/value_loss",
    "losses/entropy", "losses/approx_kl", "charts/learning_rate",
]
present = [t for t in panel_tags if t in scalar_tags]
ncol = 3
nrow = (len(present) + ncol - 1) // ncol
fig, axes = plt.subplots(nrow, ncol, figsize=(12, 3.2 * nrow))
axes = axes.ravel()
for ax, tag in zip(axes, present):
    steps, vals = series(tag)
    ax.plot(steps, vals, lw=1.2)
    ax.set_title(tag, fontsize=9)
    ax.set_xlabel("env step")
    ax.grid(alpha=0.3)
for ax in axes[len(present):]:
    ax.axis("off")
fig.suptitle("PPO training scalars", y=1.01)
plt.tight_layout()
plt.show()

### The held-out `eval/*` trajectory

The bottom line. Left: mean episode return, RL vs baseline, over training. Right:
`eval/paired_uplift` (mean per-seed RL − baseline on identical worlds) with `eval/win_rate`
overlaid. A **positive and stable** uplift is the goal; a curve that rises then degrades
is why §3 selects the *best* checkpoint rather than the last.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

s_rl, v_rl = series("eval/rl_return")
s_bl, v_bl = series("eval/baseline_return")
if len(v_rl): axes[0].plot(s_rl, v_rl, marker="o", ms=3, label="RL")
if len(v_bl): axes[0].plot(s_bl, v_bl, marker="s", ms=3, label="baseline (OrderUpTo)")
axes[0].set_title("eval return: RL vs baseline")
axes[0].set_xlabel("env step"); axes[0].set_ylabel("mean episode return")
axes[0].legend(); axes[0].grid(alpha=0.3)

s_up, v_up = series("eval/paired_uplift")
s_wr, v_wr = series("eval/win_rate")
ax = axes[1]
if len(v_up):
    ax.plot(s_up, v_up, marker="o", ms=3, color="C2", label="paired_uplift")
    ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_title("eval/paired_uplift  (>0 means RL beats baseline)")
ax.set_xlabel("env step"); ax.set_ylabel("mean (RL − baseline)")
ax.grid(alpha=0.3)
if len(v_wr):
    axb = ax.twinx()
    axb.plot(s_wr, v_wr, marker="^", ms=3, color="C3", alpha=0.6, label="win_rate")
    axb.set_ylabel("win rate"); axb.set_ylim(0, 1)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

if len(v_up):
    print(f"final logged paired_uplift : {v_up[-1]:+,.1f}")
    print(f"best  logged paired_uplift : {v_up.max():+,.1f}  (at env step {int(s_up[v_up.argmax()]):,})")

## 3 · Pick the checkpoint to analyze

The eval-cadence checkpoints line up one-for-one with the logged `eval/*` points (both fire
on the same schedule), so the checkpoint at the **best** `eval/paired_uplift` is the one
worth keeping — picking the last checkpoint is a common mistake when the curve has already
turned over. We map eval index → checkpoint and select the best, falling back to the final
checkpoint if no eval series was logged.

In [ ]:
# Eval-cadence checkpoints are named by eval index (0,1,2,…); the final checkpoint is named
# by total_env_steps and sorts last. Align the cadence checkpoints with the eval series.
cadence_ckpts = ckpts[:-1] if len(ckpts) > 1 else ckpts

if len(v_up) and len(cadence_ckpts) >= len(v_up):
    best_eval_idx = int(v_up.argmax())
    chosen_ckpt = cadence_ckpts[best_eval_idx]
    reason = (f"best eval/paired_uplift ({v_up[best_eval_idx]:+,.1f}) "
              f"at env step {int(s_up[best_eval_idx]):,}")
else:
    chosen_ckpt = ckpts[-1]
    reason = "final checkpoint (no usable eval series to select on)"

print(f"final  checkpoint : {ckpts[-1].name}")
print(f"chosen checkpoint : {chosen_ckpt.name}")
print(f"  -> {reason}")

## 4 · Deep CRN comparison vs the textbook default

We load the chosen actor and re-run the **exact** CRN evaluation `train.py` uses — but on
`N_EVAL_SEEDS` held-out seeds (more than training logged, for a tighter estimate). The
baseline is a fresh `OrderUpToPolicy()`. For each seed the RL and baseline arms share
`(world_seed, capacity, balance, active subset, slot permutation)`, so any gap is the policy
alone. We collect **per-seed** returns and KPIs by calling `evaluate(...)` one spec at a
time — the trick that exposes detail the aggregate hides.

In [ ]:
from src.rl.agents.ppo import Actor
from src.rl.configs.default import RLConfig
from src.rl.episode_sampler import make_synthetic_catalog
from src.rl.eval import build_eval_seeds, evaluate
from src.sim.policy import OrderUpToPolicy

cfg = RLConfig(K_active=K_ACTIVE, K_catalog=K_CATALOG, episode_length=EPISODE_LENGTH)

actor = Actor(observation_dim(cfg.K_active), action_dim(cfg.K_active))
actor.load_state_dict(torch.load(chosen_ckpt, map_location="cpu"))
actor.eval()

def rl_policy_fn(obs_np):
    obs_t = torch.tensor(obs_np, dtype=torch.float32).unsqueeze(0)
    with torch.no_grad():
        action, _, _ = actor.get_action_and_log_prob(obs_t)
    return action.squeeze(0).numpy()

catalog = make_synthetic_catalog(cfg.K_catalog)
eval_specs = build_eval_seeds(catalog, cfg, n_seeds=N_EVAL_SEEDS)
print(f"built {len(eval_specs)} held-out CRN eval specs")

KPIS = ["return", "net_profit", "stockout_rate",
        "inventory_turnover", "mean_price_pct_of_msrp"]

rows = []
for i, spec in enumerate(eval_specs):
    m = evaluate(rl_policy_fn, lambda: OrderUpToPolicy(), [spec], config=cfg)
    row = {"seed_idx": i, "capacity": spec.capacity, "balance": spec.balance}
    for kpi in KPIS:
        row[f"rl_{kpi}"] = m.get(f"eval/rl_{kpi}", np.nan)
        row[f"bl_{kpi}"] = m.get(f"eval/baseline_{kpi}", np.nan)
    rows.append(row)
ev = pd.DataFrame(rows)
ev["uplift"] = ev["rl_return"] - ev["bl_return"]
print(f"per-seed table: {ev.shape[0]} rows")
ev.head()

### Headline: aggregate KPIs, uplift, and a bootstrap CI

The aggregate RL-vs-baseline KPI table, plus the paired uplift with a 95% bootstrap
confidence interval over seeds. A CI **strictly above zero** is the honest claim that the
policy beats the textbook rule on held-out worlds — not just on lucky draws.

In [ ]:
agg = pd.DataFrame({
    "RL":       [ev[f"rl_{k}"].mean() for k in KPIS],
    "baseline": [ev[f"bl_{k}"].mean() for k in KPIS],
}, index=KPIS)
agg["RL − baseline"] = agg["RL"] - agg["baseline"]
display(agg.round(3))

# Paired bootstrap CI on the per-seed uplift.
rng = np.random.default_rng(0)
uplift = ev["uplift"].to_numpy()
boot = rng.choice(uplift, size=(10_000, len(uplift)), replace=True).mean(axis=1)
ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
win_rate = (uplift >= 0).mean()

print(f"\npaired uplift (mean RL − baseline net profit/seed) : {uplift.mean():+,.1f}")
print(f"95% bootstrap CI                                   : [{ci_lo:+,.1f}, {ci_hi:+,.1f}]")
print(f"win rate (seeds where RL >= baseline)              : {win_rate:.0%}  "
      f"({int((uplift >= 0).sum())}/{len(uplift)})")
verdict = ("a real improvement over the default" if ci_lo > 0 else
           "NOT clearly better than the default (CI spans 0)")
print(f"\nVerdict: the trained policy is {verdict}.")

### The per-seed picture

Left: every held-out seed as (baseline, RL) on the 45° line — points **above** the line are
RL wins. Outliers far below the line are regimes the policy mishandles. Right: the
distribution of per-seed uplift, with its mean.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

lo = min(ev["bl_return"].min(), ev["rl_return"].min())
hi = max(ev["bl_return"].max(), ev["rl_return"].max())
axes[0].plot([lo, hi], [lo, hi], color="k", lw=0.8, ls="--", zorder=1)
colors = np.where(ev["uplift"] >= 0, "#55A868", "#C44E52")
axes[0].scatter(ev["bl_return"], ev["rl_return"], c=colors, s=35, alpha=0.8, zorder=3)
axes[0].set_xlabel("baseline return"); axes[0].set_ylabel("RL return")
axes[0].set_title("per-seed: RL vs baseline (above line = RL wins)")
axes[0].grid(alpha=0.3)

axes[1].hist(uplift, bins=20, color="#4C72B0", edgecolor="white")
axes[1].axvline(0, color="k", lw=0.8, ls="--")
axes[1].axvline(uplift.mean(), color="#C44E52", lw=2, label=f"mean {uplift.mean():+,.0f}")
axes[1].set_xlabel("per-seed uplift (RL − baseline)"); axes[1].set_ylabel("# seeds")
axes[1].set_title("uplift distribution")
axes[1].legend()
plt.tight_layout()
plt.show()

## 5 · Regime analysis — does the scale-invariance hold?

Each episode draws a node *capacity* and opening *balance* **log-uniform across two orders
of magnitude**, and the whole point of the scale-invariance package (order-up-to action
decoder, demand-units inventory feature) is that *one* policy generalises across that range.
Here we slice the per-seed uplift by capacity and balance to see **where** the policy wins
or loses — a flat, positive band across all sizes is the result you want; a slope means the
policy is tuned to one end of the range.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, col, label in [(axes[0], "capacity", "node capacity"),
                       (axes[1], "balance", "opening balance")]:
    colors = np.where(ev["uplift"] >= 0, "#55A868", "#C44E52")
    ax.scatter(ev[col], ev["uplift"], c=colors, s=35, alpha=0.8)
    ax.axhline(0, color="k", lw=0.8, ls="--")
    ax.set_xscale("log")
    ax.set_xlabel(f"{label} (log scale)"); ax.set_ylabel("per-seed uplift")
    ax.set_title(f"uplift vs. {label}")
    ax.grid(alpha=0.3)

    # binned means (terciles on the log axis) to read the trend through the noise.
    q = pd.qcut(ev[col], q=3, duplicates="drop")
    binned = ev.groupby(q, observed=True)["uplift"].mean()
    for interval, val in binned.items():
        ax.hlines(val, interval.left, interval.right, color="#1f1f1f", lw=2.5, zorder=4)
plt.tight_layout()
plt.show()

print("mean uplift by capacity tercile:")
print(ev.groupby(pd.qcut(ev["capacity"], 3, duplicates="drop"),
                 observed=True)["uplift"].mean().round(1).to_string())

## 6 · Behavioral signature — *how* does the policy differ?

Beyond the bottom line: the per-seed KPI distributions show *how* the learned policy departs
from the textbook rule. A lower stockout rate at comparable turnover means it ordered
better; the price-%-of-MSRP panel surfaces whether the agent is pulling its profit from
*pricing* rather than from *ordering*. Box pairs are RL (blue) vs baseline (orange) across
all held-out seeds.

> The CRN evaluator (`src/rl/eval.py`) reports the KPIs recoverable from observable traces —
> return, net profit, stockout rate, turnover, and price % of MSRP — so those are what we
> compare here.

In [ ]:
behavior_kpis = ["stockout_rate", "inventory_turnover", "mean_price_pct_of_msrp"]
fig, axes = plt.subplots(1, len(behavior_kpis), figsize=(3.2 * len(behavior_kpis), 4))
for ax, kpi in zip(axes, behavior_kpis):
    data = [ev[f"rl_{kpi}"].dropna(), ev[f"bl_{kpi}"].dropna()]
    bp = ax.boxplot(data, patch_artist=True, widths=0.6)
    for box, c in zip(bp["boxes"], ["#4C72B0", "#DD8452"]):
        box.set_facecolor(c); box.set_alpha(0.7)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["RL", "base"])
    ax.set_title(kpi, fontsize=9)
    ax.grid(axis="y", alpha=0.3)
fig.suptitle("behavioral signature — RL vs textbook default across held-out seeds", y=1.02)
plt.tight_layout()
plt.show()

print("mean by KPI (RL vs baseline):")
display(agg.loc[behavior_kpis].round(3))

## Where to next

- Re-point `RUN_DIR` at any other `runs/<experiment>/` to compare sweeps — overlay their
  `eval/paired_uplift` curves in TensorBoard (`uv run tensorboard --logdir runs/`) to pick
  which run to deep-dive here.
- Train longer or on a real catalog: add `--setup-dir setups/<name>` to the training command
  (then load that setup's catalog here instead of the synthetic one for the eval to match).
- For the strongest textbook bar, swap the baseline `OrderUpToPolicy()` for the **tuned**
  winner from notebook **05a**'s study (`OrderUpToPolicy(**best_params)`), and re-run §4 —
  the CRN guarantee still holds.
- `src/rl/README.md` — observation/action layout, checkpoint format, and the CRN eval contract.

| notebook | topic |
|---|---|
| **06** | train a PPO agent & evaluate it — *how to drive the stack* |
| **06a** | deep analysis of a long-trained run from disk — *this notebook* |
| **05a** | analyze a completed tuning study (the baseline this agent is measured against) |